# Correlation Analysis & City Clustering - Air Quality Analyzer
This notebook analyzes correlation matrices between air pollutants and performs KMeans clustering on cities.

## Section 1: Load Data

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans

clean_path = os.path.join('..', 'data', 'processed', 'aqi_clean.csv')
if not os.path.exists(clean_path):
    clean_path = os.path.join('data', 'processed', 'aqi_clean.csv')

df_clean = pd.read_csv(clean_path)
print(f"Loaded clean data shape: {df_clean.shape}")

## Section 2: Pollutant Correlation Matrix

In [ ]:
pollutant_cols = ['pm25', 'pm10', 'no2', 'co']
corr_matrix = df_clean[pollutant_cols].corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    labels=dict(x="Pollutant", y="Pollutant", color="Correlation"),
    x=pollutant_cols,
    y=pollutant_cols,
    color_continuous_scale="RdBu_r",
    title="Pollutant Correlation Matrix"
)
fig_corr.show()

# Find highest correlation pair (excluding self-correlation)
unstack_corr = corr_matrix.unstack()
unstack_corr = unstack_corr[unstack_corr < 1.0]
if not unstack_corr.empty:
    max_pair = unstack_corr.idxmax()
    max_val = unstack_corr.max()
    print(f"Most correlated pollutants: {max_pair[0].upper()} and {max_pair[1].upper()} (r = {max_val:.2f})")

## Section 3: AQI vs Time of Day

In [ ]:
if 'hour' in df_clean.columns:
    hourly_aqi = df_clean.groupby('hour')['pm25'].mean().reset_index()
    fig_hour = px.line(hourly_aqi, x='hour', y='pm25', title="Average PM2.5 by Hour of Day")
    fig_hour.show()
else:
    print("Hour column does not exist in dataset — snapshot measurements only.")

## Section 4: City Clustering (KMeans)

In [ ]:
city_features = df_clean.groupby('city')[pollutant_cols].mean().dropna()

# Apply KMeans with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
city_features['cluster_id'] = kmeans.fit_predict(city_features[pollutant_cols])

# Map Cluster Labels
cluster_map = {
    0: "Industrial pollution dominant",
    1: "Mixed pollution",
    2: "Relatively clean"
}
city_features['cluster_label'] = city_features['cluster_id'].map(cluster_map)

# Update city_risk_ranking.csv
risk_path = os.path.join('..', 'data', 'processed', 'city_risk_ranking.csv')
if not os.path.exists(risk_path):
    risk_path = os.path.join('data', 'processed', 'city_risk_ranking.csv')

if os.path.exists(risk_path):
    city_risk_df = pd.read_csv(risk_path)
    city_risk_df = city_risk_df.merge(city_features[['cluster_label']], on='city', how='left')
    city_risk_df['cluster_label'] = city_risk_df['cluster_label'].fillna("Mixed pollution")
    city_risk_df.to_csv(risk_path, index=False)
    print("Updated city_risk_ranking.csv with cluster_label column.")
    display(city_risk_df.head(10))